# 01 – Khám phá dữ liệu (EDA)

**EDA (Exploratory Data Analysis)** là bước nhìn vào dữ liệu *trước khi huấn luyện*. Nếu dữ liệu có vấn đề (lệch nhãn, ảnh sai nhãn, ảnh quá nhỏ...) thì dù model tốt đến đâu kết quả cũng không đáng tin.

**Notebook này trả lời 4 câu hỏi:**
1. Có bao nhiêu ảnh, chia theo từng loại trái cây và theo tập train/test?
2. Hai nhãn `fresh` và `rotten` có cân bằng không?
3. Ảnh trông như thế nào (nền, góc chụp, ánh sáng)? Nhãn có đúng không?
4. Kích thước ảnh có đồng đều không?

**Cách chạy:** bấm *Run All*, hoặc chạy lần lượt từng ô từ trên xuống. Notebook chỉ đọc dữ liệu, không thay đổi gì trong `data/raw/`.

## 0. Chuẩn bị và đọc dữ liệu

- `sys.path.insert(0, ...)`: notebook nằm trong `notebooks/` nên cần trỏ về thư mục gốc dự án để `import src...` được.
- `build_dataframe(thư_mục)` quét toàn bộ ảnh và trả về bảng (DataFrame) có 3 cột:
  - `filename`: đường dẫn ảnh
  - `label`: nhãn gộp, chỉ có `fresh` hoặc `rotten` (suy ra từ tên thư mục, ví dụ `freshapples` → `fresh`)
  - `source`: tên thư mục gốc, tức loại trái cây + trạng thái (ví dụ `rottenbanana`)
- Hai biến `train_df` và `test_df` là danh sách ảnh của tập train và tập test.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # trỏ về thư mục gốc để import được package src

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.data_loader import TRAIN_DIR, TEST_DIR, build_dataframe

train_df = build_dataframe(TRAIN_DIR)  # quét data/raw/train
test_df = build_dataframe(TEST_DIR)    # quét data/raw/test
print('Train:', len(train_df), 'ảnh | Test:', len(test_df), 'ảnh')
train_df.head()

## 1. Số ảnh theo thư mục và theo nhãn

**Mục đích:** kiểm tra dữ liệu có cân bằng không.

- Bảng đầu tiên: số ảnh của từng loại (`source`) ở tập train và test. Biểu đồ cột vẽ lại cùng số liệu cho dễ so sánh.
- Bảng thứ hai: số ảnh của hai nhãn `fresh` và `rotten`.

**Cách đọc:** nếu một nhãn nhiều hơn hẳn nhãn kia (ví dụ gấp 3 lần), model sẽ có xu hướng đoán nhãn nhiều hơn và con số *accuracy* trở nên đánh lừa. Khi đó nên thêm ảnh cho nhãn ít hoặc dùng class weight.

Cũng nên để ý: tập test có số ảnh đủ lớn cho từng loại không, và tỉ lệ train/test có hợp lý không.

In [ ]:
# Gộp train và test vào một bảng, thêm cột split để phân biệt
all_df = pd.concat([train_df.assign(split='train'), test_df.assign(split='test')])

# Đếm số ảnh theo từng loại và từng tập
by_source = all_df.groupby(['source', 'split']).size().unstack('split')
display(by_source)
by_source.plot(kind='bar', figsize=(9, 4), title='Số ảnh theo thư mục')
plt.tight_layout()
plt.show()

# Đếm số ảnh theo nhãn fresh / rotten
display(all_df.groupby(['split', 'label']).size().unstack('label'))

## 2. Ảnh mẫu (2 ảnh mỗi loại)

**Mục đích:** nhìn bằng mắt để kiểm tra 3 thứ.

1. **Nhãn có đúng không?** Ảnh trong thư mục `rotten...` có thật sự bị hỏng không? Nhãn sai là nguyên nhân phổ biến làm model học kém.
2. **Kiểu ảnh:** một quả hay nhiều quả, nền đơn giản hay lộn xộn, có tay cầm không. Model chỉ giỏi với kiểu ảnh giống dữ liệu huấn luyện. Nếu toàn ảnh nền sạch thì model có thể đoán sai với ảnh thực tế (tay cầm, nền phức tạp).
3. **Chất lượng:** ảnh có bị mờ, quá tối, bị cắt mất quả không.

Mỗi lần chạy lấy 2 ảnh cố định (`random_state=0`) nên kết quả giống nhau giữa các lần. Đổi số này để xem ảnh khác.

In [ ]:
sources = sorted(train_df['source'].unique())  # danh sách các loại, ví dụ freshapples, rottenbanana...
fig, axes = plt.subplots(2, len(sources), figsize=(3 * len(sources), 6))
for j, s in enumerate(sources):
    # lấy ngẫu nhiên 2 ảnh của loại s
    files = train_df[train_df['source'] == s]['filename'].sample(2, random_state=0)
    for i, f in enumerate(files):
        axes[i][j].imshow(Image.open(f).convert('RGB'))
        axes[i][j].axis('off')
        if i == 0:
            axes[i][j].set_title(s)  # tên loại ở hàng trên cùng
plt.tight_layout()
plt.show()

## 3. Kích thước ảnh

**Mục đích:** biết ảnh gốc to hay nhỏ, vuông hay chữ nhật.

Trước khi vào model, mọi ảnh đều bị resize về **224 × 224**. Vì vậy:
- Ảnh rất nhỏ (ví dụ dưới 100 px) bị phóng to nên mờ, làm model khó nhận ra đặc điểm.
- Ảnh có tỉ lệ rất lệch (ví dụ dài gấp 3 lần rộng) sẽ bị méo khi ép về hình vuông.

Bảng `describe()` cho biết min, max, trung bình của chiều rộng (`width`) và chiều cao (`height`). Để chạy nhanh, chỉ lấy mẫu tối đa 300 ảnh.

In [ ]:
sample = train_df['filename'].sample(min(300, len(train_df)), random_state=0)
sizes = pd.DataFrame([Image.open(f).size for f in sample], columns=['width', 'height'])
display(sizes.describe())  # count, mean, std, min, 25%, 50%, 75%, max

## Tổng kết: cần rút ra gì từ notebook này?

| Câu hỏi | Dấu hiệu tốt | Dấu hiệu cần xử lý |
|---|---|---|
| Cân bằng nhãn | `fresh` và `rotten` có số ảnh gần nhau | Một nhãn nhiều hơn hẳn → thêm ảnh hoặc dùng class weight |
| Nhãn đúng | Ảnh mẫu khớp với tên thư mục | Có ảnh sai nhãn → sửa hoặc xóa |
| Đa dạng ảnh | Có nhiều nền, góc chụp, ánh sáng | Toàn ảnh nền sạch → model yếu với ảnh thực tế, nên bổ sung ảnh thật |
| Kích thước | Phần lớn ảnh đủ lớn (vài trăm px) | Nhiều ảnh quá nhỏ → loại bỏ |

Sau khi kiểm tra xong, chuyển sang notebook **02** (mô hình baseline).